In [ ]:
# ============================================================
# HSTからシンク表面の降着率 Mdot_flux を読み、
# 時間 [kyr]－降着率 [M_sun yr^-1] を描画するコード
#
# HSTヘッダは1始まり：
#   [1]  = time
#   [14] = Mdot_flux
#
# NumPy配列は0始まり：
#   data[:, 0]  = time
#   data[:, 13] = Mdot_flux
# ============================================================

import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# 入出力設定
# ------------------------------------------------------------
hst_file = Path(
    os.path.expanduser(
        "~/athena-project/results/〇〇/Toyouchi.hst"
    )
).resolve()

output_dir = Path(
    "./stellar_mass_rate_history"
).resolve()

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_file = output_dir / "mdot_flux_vs_time.png"


# ------------------------------------------------------------
# 新しいコード単位
# L0 = rb / 2 の単位系
# ------------------------------------------------------------
M_UNIT_CGS = 4.0e33       # 1 code mass [g]
T_UNIT_CGS = 3.61e10      # 1 code time [s]

M_SUN_CGS = 1.98847e33    # 太陽質量 [g]
YEAR_CGS = 365.25 * 24.0 * 3600.0
KYR_CGS = 1.0e3 * YEAR_CGS

# 時間：code time → kyr
CODE_TIME_TO_KYR = (
    T_UNIT_CGS / KYR_CGS
)

# 降着率：
# code mass / code time → M_sun / yr
CODE_MDOT_TO_MSUN_PER_YR = (
    M_UNIT_CGS / M_SUN_CGS
) / (
    T_UNIT_CGS / YEAR_CGS
)

print(
    f"1 code time = "
    f"{CODE_TIME_TO_KYR:.6e} kyr"
)

print(
    "1 code mass-rate = "
    f"{CODE_MDOT_TO_MSUN_PER_YR:.6e} "
    "M_sun yr^-1"
)


# ------------------------------------------------------------
# HSTヘッダから列番号を取得
# ------------------------------------------------------------
def read_hst_column_map(filename):
    """
    Athena++ HSTヘッダの
        [1]=time [2]=dt ... [14]=Mdot_flux
    を読み取る。

    戻り値はNumPy用の0始まり添字。
    """
    column_map = {}

    with open(
        filename,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as handle:

        for line in handle:
            if not line.lstrip().startswith("#"):
                continue

            matches = re.findall(
                r"\[\s*(\d+)\s*\]\s*=\s*([^\s]+)",
                line,
            )

            for number_text, name in matches:
                hst_number = int(number_text)

                # HSTの1始まり番号を
                # NumPyの0始まり番号へ変換
                column_map[name] = hst_number - 1

    return column_map


column_map = read_hst_column_map(
    hst_file
)

print("\n[INFO] Detected HST columns:")

for name, numpy_index in sorted(
    column_map.items(),
    key=lambda item: item[1],
):
    print(
        f"       HST [{numpy_index + 1:2d}] "
        f"-> NumPy [{numpy_index:2d}] = {name}"
    )


if "time" not in column_map:
    raise KeyError(
        "HSTヘッダからtime列を取得できませんでした。"
    )

if "Mdot_flux" not in column_map:
    raise KeyError(
        "HSTヘッダからMdot_flux列を取得できませんでした。"
    )

time_column = column_map["time"]
mdot_column = column_map["Mdot_flux"]

print(
    f"\n[INFO] time      : "
    f"data[:, {time_column}]"
)

print(
    f"[INFO] Mdot_flux : "
    f"data[:, {mdot_column}]"
)


# ------------------------------------------------------------
# データ読み込み
# ------------------------------------------------------------
data = np.loadtxt(
    hst_file,
    comments="#",
    ndmin=2,
)

required_column = max(
    time_column,
    mdot_column,
)

if data.shape[1] <= required_column:
    raise ValueError(
        "HSTの列数が不足しています。"
        f" columns={data.shape[1]}, "
        f"required index={required_column}"
    )

time_code_raw = np.asarray(
    data[:, time_column],
    dtype=float,
)

mdot_code_raw = np.asarray(
    data[:, mdot_column],
    dtype=float,
)

finite_mask = (
    np.isfinite(time_code_raw)
    & np.isfinite(mdot_code_raw)
)

time_code_raw = time_code_raw[
    finite_mask
]

mdot_code_raw = mdot_code_raw[
    finite_mask
]

if time_code_raw.size == 0:
    raise RuntimeError(
        "有限なtime/Mdot_fluxデータがありません。"
    )


# ------------------------------------------------------------
# リスタート重複行の整理
#
# Athena++のHSTはリスタート時に、古い時刻へ戻った
# データがファイル末尾へ追記されることがある。
#
# 後ろから走査し、最後に書かれた時間系列を優先する。
# Mdot_fluxの増減ではリスタート判定しない。
# ------------------------------------------------------------
def keep_latest_restart_branch(
    time_values,
    *value_arrays,
):
    time_values = np.asarray(
        time_values,
        dtype=float,
    )

    keep_reversed = []
    latest_accepted_time = np.inf

    for index in range(
        len(time_values) - 1,
        -1,
        -1,
    ):
        current_time = time_values[index]

        scale = max(
            1.0,
            abs(current_time),
            (
                abs(latest_accepted_time)
                if np.isfinite(latest_accepted_time)
                else 1.0
            ),
        )

        tolerance = (
            64.0
            * np.finfo(float).eps
            * scale
        )

        if (
            current_time
            < latest_accepted_time - tolerance
        ):
            keep_reversed.append(index)
            latest_accepted_time = current_time

        elif np.isclose(
            current_time,
            latest_accepted_time,
            rtol=0.0,
            atol=tolerance,
        ):
            # 同一時刻が重複した場合は、
            # 後に書かれた行を優先する
            continue

    keep_indices = np.asarray(
        keep_reversed[::-1],
        dtype=int,
    )

    cleaned_time = time_values[
        keep_indices
    ]

    cleaned_values = [
        np.asarray(values)[keep_indices]
        for values in value_arrays
    ]

    return (
        cleaned_time,
        cleaned_values,
        keep_indices,
    )


(
    time_code,
    cleaned_arrays,
    keep_indices,
) = keep_latest_restart_branch(
    time_code_raw,
    mdot_code_raw,
)

mdot_code = cleaned_arrays[0]

print(
    f"\n[INFO] Raw rows     : "
    f"{len(time_code_raw)}"
)

print(
    f"[INFO] Cleaned rows : "
    f"{len(time_code)}"
)

print(
    f"[INFO] Removed rows : "
    f"{len(time_code_raw) - len(time_code)}"
)

if len(time_code) < 2:
    raise RuntimeError(
        "リスタート整理後のデータが2行未満です。\n"
        "time列とHSTファイル内の時刻配列を"
        "確認してください。"
    )


# ------------------------------------------------------------
# 物理単位へ変換
# ------------------------------------------------------------
time_kyr = (
    time_code
    * CODE_TIME_TO_KYR
)

mdot_msun_per_yr = (
    mdot_code
    * CODE_MDOT_TO_MSUN_PER_YR
)


# ------------------------------------------------------------
# 平均降着率を計算
# ------------------------------------------------------------

# 単純な標本平均
mdot_sample_mean = np.mean(
    mdot_msun_per_yr
)

# 時間重み付き平均
#
# <Mdot>_t =
# integral Mdot(t) dt / (t_final - t_initial)
#
# 出力時間間隔が一定でない場合はこちらが適切。
time_yr = time_kyr * 1.0e3
elapsed_time_yr = (
    time_yr[-1] - time_yr[0]
)

if elapsed_time_yr > 0.0:
    if hasattr(np, "trapezoid"):
        integrated_mass_msun = np.trapezoid(
            mdot_msun_per_yr,
            x=time_yr,
        )
    else:
        # 古いNumPy向け
        integrated_mass_msun = np.trapz(
            mdot_msun_per_yr,
            x=time_yr,
        )

    mdot_time_mean = (
        integrated_mass_msun
        / elapsed_time_yr
    )
else:
    integrated_mass_msun = np.nan
    mdot_time_mean = np.nan

mdot_min = np.min(
    mdot_msun_per_yr
)

mdot_max = np.max(
    mdot_msun_per_yr
)

negative_count = np.count_nonzero(
    mdot_msun_per_yr < 0.0
)


# ------------------------------------------------------------
# 診断表示
# ------------------------------------------------------------
print(
    f"\n[INFO] Time range = "
    f"{time_kyr[0]:.6e} -- "
    f"{time_kyr[-1]:.6e} kyr"
)

print(
    f"[INFO] Mdot range = "
    f"{mdot_min:.6e} -- "
    f"{mdot_max:.6e} M_sun yr^-1"
)

print(
    f"[INFO] Time-weighted mean Mdot = "
    f"{mdot_time_mean:.6e} M_sun yr^-1"
)

print(
    f"[INFO] Sample mean Mdot        = "
    f"{mdot_sample_mean:.6e} M_sun yr^-1"
)

print(
    f"[INFO] Integral of Mdot        = "
    f"{integrated_mass_msun:.6e} M_sun"
)

print(
    f"[INFO] Negative Mdot rows      = "
    f"{negative_count}/{len(mdot_msun_per_yr)}"
)


# ------------------------------------------------------------
# プロット
# ------------------------------------------------------------
fig, ax = plt.subplots(
    figsize=(9, 6),
)

ax.plot(
    time_kyr,
    mdot_msun_per_yr,
    color="tab:blue",
    linewidth=2.0,
    label=r"$\dot{M}_{\rm flux}$",
)

# 時間平均を水平線で表示
ax.axhline(
    mdot_time_mean,
    color="tab:red",
    linewidth=1.5,
    linestyle="--",
    label="Time-weighted mean",
)

ax.axhline(
    0.0,
    color="black",
    linewidth=0.8,
    alpha=0.7,
)

ax.set_xlabel(
    "Time [kyr]"
)

ax.set_ylabel(
    r"$\dot{M}_{\rm flux}$ "
    r"$[M_\odot\,{\rm yr}^{-1}]$"
)

ax.set_title(
    "Accretion rate through the sink surface"
)

ax.grid(
    True,
    linestyle="--",
    alpha=0.45,
)

ax.legend(
    loc="best",
)

# 降着率の変動幅が非常に大きく、
# 正負の両方を対数的に見たい場合：
# ax.set_yscale("symlog", linthresh=1.0e-6)

fig.tight_layout()

fig.savefig(
    output_file,
    dpi=200,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(
    f"\n[SAVED] {output_file}"
)